# TIMEOrdinalDAG · GridView / spacetime
选择 spacetime 内核，修改 PROJECT 与 BASE。先执行小规模验证，完整模拟和真实数据默认关闭。Python 参考实现与 R/BiDAG 实现不同。本包不含论文、审稿文件、输入数据或历史实验结果。

In [ ]:
import os, sys, subprocess, json
from pathlib import Path
PROJECT = Path.cwd()  # 改成解压后包含 experiments 的目录
BASE = PROJECT.parent / "TIMEOrdinalDAG_my_run"  # 改成集群持久化存储目录
WORKERS = 4  # 不超过本次作业实际分配的 CPU 数
assert (PROJECT / "experiments/run.py").is_file(), "请修改 PROJECT"
BASE.mkdir(parents=True, exist_ok=True)
ENV = os.environ.copy()
ENV.update(OPENBLAS_NUM_THREADS="1", OMP_NUM_THREADS="1", MKL_NUM_THREADS="1", MPLBACKEND="Agg",
           MPLCONFIGDIR=str(BASE/".cache/matplotlib"), XDG_CACHE_HOME=str(BASE/".cache"))
Path(ENV["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
PYTHON = sys.executable
print("当前内核:", PYTHON, sys.version)
def run(args):
    subprocess.run(list(map(str,args)), cwd=PROJECT/"experiments", env=ENV, check=True)


## 安装依赖
使用当前内核解释器。默认范围依赖适合已有 spacetime 环境；Python 3.12 的全新环境可选 environment_lock.txt。安装完成后如数值包被更新，请重启内核，重新运行配置单元，跳过安装单元。

In [ ]:
INSTALL = False  # 首次运行设 True；安装后改回 False
if INSTALL:
    run([PYTHON, "-m", "pip", "install", "-r", PROJECT/"experiments/requirements.txt"])


## 测试和小规模模拟（不需要外部数据）

In [ ]:
run([PYTHON,"-m","unittest","discover","-s",PROJECT/"experiments","-p","test_revision.py","-v"])
run([PYTHON,"checks.py"])
run([PYTHON,"run.py","simulate","--backend","python-reference","--out",BASE/"pilot",
     "--subjects","20","--length","60","--p","2","--lag","1","--penalties","6",
     "--mc","3","--max-em","4","--bootstrap","3","--baselines","pcmci,pcmciplus,lpcmci"])


In [ ]:
import pandas as pd
summary = pd.read_csv(BASE/"pilot/simulation_summary.csv")
assert not summary.get("status",pd.Series(dtype=str)).eq("FAILED").any(), summary
print(summary.to_string(index=False))
print(pd.read_csv(BASE/"pilot/rep_000/OCE_results.csv").to_string(index=False))
baselines = pd.read_csv(BASE/"pilot/rep_000/baselines.csv")
assert baselines["status"].eq("ok").all(), baselines
print(baselines.to_string(index=False))


## 完整模拟（长任务，需主动启用）
默认关闭，避免 Run All 意外启动长计算。完整计划含 1,650 次设置内运行，两组覆盖率实验各 100×500 次 bootstrap。请在集群计算节点申请足够时限。建议先申请 4 CPU、16 GB 内存作为起点，再根据 pilot 监控调整；不需要 GPU。Jupyter 会话/资源到期会终止任务，Popen 不能绕过集群回收。

In [ ]:
RUN_FULL = False
RESULTS = BASE/"results"
if RUN_FULL:
    if "full_process" in globals() and full_process.poll() is None:
        raise RuntimeError("本内核已有任务在运行")
    log = BASE/"study.log"
    with log.open("a") as f:
        full_process = subprocess.Popen([PYTHON,"-u","study.py","--out",str(RESULTS),"--workers",str(WORKERS)],
            cwd=PROJECT/"experiments",env=ENV,stdout=f,stderr=subprocess.STDOUT)
    print("PID", full_process.pid, "日志", log)


In [ ]:
for name in ["progress.json", "environment.json"]:
    p=RESULTS/name
    if p.exists():
        obj=json.loads(p.read_text())
        print(name, len(obj) if isinstance(obj,list) else obj)
p=BASE/"study.log"
if p.exists(): print("\n".join(p.read_text(errors="replace").splitlines()[-12:]))
if "full_process" in globals(): print("退出码（None 表示仍在运行）:",full_process.poll())


In [ ]:
SUMMARIZE = False  # 所有模拟结束且无 FAILED 后设 True
if SUMMARIZE:
    plan=json.loads((RESULTS/"plan.json").read_text())
    assert plan, "计划为空"
    for job in plan:
        p=Path(job["out"])/"status.json"
        assert p.exists() and json.loads(p.read_text())["status"] == "complete", str(p)
    run([PYTHON,"report_study.py","--results",RESULTS,"--out",BASE/"summary"])
    run([PYTHON,"plot_regimes.py","--results",RESULTS,"--out",BASE/"plots"])
    run([PYTHON,"audit_outputs.py","--results",RESULTS,"--out",BASE/"summary/artifact_validation.json"])


## 可选：真实 Fitbit 数据
原始 CSV 不在本包内。只使用同一 April–May 导出期：heartrate_seconds_merged.csv 和 minuteStepsNarrow.csv（或 minuteStepsNarrow_merged.csv）。设置 RAW 为 CSV 所在目录。先小 bootstrap 验证，再用新的输出目录跑 500 次正式分析。

In [ ]:
RUN_REAL = False
RAW = BASE/"raw"
REAL_BOOTSTRAP = 3  # 首次验证用 3，正式用 500，并换新的 REAL_OUT
REAL_OUT = BASE/"real_pilot"
if RUN_REAL:
    run([PYTHON,"prepare_fitbit.py","--data",RAW,"--out",BASE/"prepared"])
    run([PYTHON,"run.py","real","--backend","python-reference","--input",BASE/"prepared/minute_panel.csv",
         "--out",REAL_OUT,"--columns","Steps,HR","--calibrate","--lag","2","--q","4",
         "--scheme","cart","--penalties","2,6,10","--mc","20","--bootstrap",str(REAL_BOOTSTRAP),
         "--stride","10","--loo","--baselines",""])
    run([PYTHON,"diagnostics.py","--input",BASE/"prepared/minute_panel.csv","--fit",REAL_OUT,"--columns","Steps,HR"])


## 中断和解释
重启内核后先检查集群中旧任务是否仍在运行，每个输出目录同时只运行一个任务。完整 study 可重用参数相符的 complete 目录；FAILED 不会自动重跑，先检查 execution.log，诊断后将失败目录移到结果根目录以外再重试。单次真实数据 bootstrap 不支持中间恢复。计算完成、图稳定、统计区间有效性是不同判断。小规模模拟仅验证软件运行，不能用作正式研究结论。